----
<div style="display: flex; align-items: center;">
    <img src="https://developers.lseg.com/content/dam/devportal/icons/logo/lseg-logo.svg" width="20%" style="vertical-align: top;">
    <h1 style="margin-left: 20px;">Data Library for Python</h1>
</div>

----

## Discovery layer - Universe Expanders
This notebook covers the public `Chain`, `Peers` and `Screener` classes.
Expanders resolve an expression or chain name into an iterable instrument universe. Constructing an object does not necessarily fetch data immediately: iteration and properties can trigger a request. Resolved universes are cached in the object; create a new object when a fresh membership lookup is required.

**Prerequisites:** a configured session with access to the underlying chain, pricing, screening and fundamental data. The peer and screener examples use desktop sessions. Chain tests also exercise platform sessions with different pricing scopes. Membership changes over time, so these examples do not assume fixed lists or counts. No orders are submitted.

#### Learn more
Visit the [LSEG Developer Portal](https://developers.lseg.com/en) for learning materials.

#### Getting Help and Support
For API questions, visit the [LSEG Developer Community](https://community.developers.lseg.com/).

----

## Set the configuration file location
Run the notebook from its own directory. Keep credentials in the configuration file.

In [ ]:
import os
os.environ["LD_LIB_CONFIG_PATH"] = "../../../Configuration"

## Some Imports to start with

In [ ]:
import lseg.data as ld
from lseg.data.discovery import Chain, Peers, Screener

## Open the data session
Configure the default session as a desktop session for the peer and screener examples.

In [ ]:
ld.open_session()

## Expand a chain
The chain tests use `0#.FCHI`. `name` identifies the chain, `constituents` provides the resolved instruments and `summary_links` contains available summary links. Iterating the chain yields constituents.

The library chooses the retrieval route according to session capabilities and scopes. The tests exercise `trapi.streaming.pricing.read` and `trapi.data.pricing.read`; summary links can depend on the retrieval route. This notebook does not change authentication scopes.

In [ ]:
chain = Chain(name="0#.FCHI")
print("Chain:", chain.name)
print("Constituents:", chain.constituents)
print("Summary links:", chain.summary_links)
list(chain)

## Resolve a peer group
`Peers(expression="LSEG.L")` builds the peer expression for the supplied instrument. Use the public iterable interface rather than accessing the internal universe cache. The test explicitly avoids a fixed peer list because classification and membership evolve.

In [ ]:
peers = Peers(expression="LSEG.L")
print(peers.expression)
peer_rics = list(peers)
peer_rics

### Use the expander as a data-request universe
As in the test, pass the `Peers` object directly to `ld.get_data` to retrieve bid and ask data. Avoid sending a request with an empty universe; an empty resolution is reported explicitly.

In [ ]:
if not peer_rics:
    raise RuntimeError("No peers resolved for LSEG.L; check data availability before requesting prices")
peer_prices = ld.get_data(universe=peers, fields=["BID", "ASK"])
peer_prices

## Screen an equity universe
`Screener` wraps a screening expression in `screen(...)`. Supply the inner expression, not an already wrapped expression. This test example screens active, public, primary equities by headquarters country code `AR` and GICS industry code `401010`.

The syntax is the screening service's expression language, not the Search API's `filter` syntax.

In [ ]:
expression = (
    'U(IN(Equity(active,public,primary))/*UNV:Public*/), '
    'IN(TR.HQCountryCode,"AR"), IN(TR.GICSIndustryCode,"401010")'
)
screener = Screener(expression=expression)
print(screener.expression)
screened_rics = list(screener)
screened_rics

### Retrieve reference and pricing fields
The test combines fundamental fields and `BID` for the screened universe.

In [ ]:
if not screened_rics:
    raise RuntimeError("The screening expression resolved no instruments; review its criteria")
screened_data = ld.get_data(
    universe=screener,
    fields=[
        "TR.CommonName", "TR.HeadquartersCountry", "TR.GICSSector",
        "TR.OrganizationStatusCode", "TR.Revenue", "BID",
    ],
)
screened_data

## Error behavior
The test also checks invalid chain names and invalid peer or screener expressions. Resolution errors are raised as `LDError` when the object is evaluated, for example by `list(Chain("invalid"))`. Construction alone is not proof that an expression is valid. Inspect reported errors rather than substituting an empty or fabricated universe.

## Close the session
Run this cell even if an earlier request fails. No persistent resources or files were created.

In [ ]:
ld.close_session()